# AAPL Option Picker

This notebook uses Quant Warehouse AAPL price history to generate oracle trade windows, then builds option rows from Quant Warehouse / ThetaData snapshots and trains a group-aware ranker to score the best contract inside each trade window.

The rebuild path tries to download missing ThetaData days when credentials are configured. If that is not available, the notebook falls back to the cached AAPL option sample in `quant-warehouse/research/options_eda_output`.


In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

repo_root = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "quant_orchestrator").is_dir()
)
warehouse_root = repo_root.parent / "quant-warehouse"
if not (warehouse_root / "quant_warehouse").is_dir():
    raise RuntimeError(f"Could not find quant-warehouse next to {repo_root}")

for path in (repo_root, warehouse_root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from quant_orchestrator.data import load_ohlcv
from quant_warehouse.target_engineering import (
    LabelBuildSpec,
    OptionLabelSpec,
    OptionMlDatasetSpec,
    ThetaDataDownloadSpec,
    build_option_ml_dataset,
    build_trade_results,
)

ARTIFACT_DIR = repo_root / "artifacts" / "aapl_option_picker"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo_root={repo_root}")
print(f"warehouse_root={warehouse_root}")
print(f"artifact_dir={ARTIFACT_DIR}")


In [ ]:
SYMBOL = "AAPL"
TRADE_START = "2015-01-01"
TRADE_END = None
DOWNLOAD_MISSING_SNAPSHOTS = True
MAX_DTE = 60
STRIKE_RANGE = 12
RANDOM_STATE = 1337

prices = load_ohlcv(SYMBOL, start=TRADE_START, end=TRADE_END).copy()
prices.index = prices.index.tz_localize(None)

label_spec = LabelBuildSpec(
    k_params={"M": [1], "QE": [1], "YE": [1]},
    min_profit_pct=0.0,
    buy_execution="high",
    sell_execution="low",
    short_execution="low",
    cover_execution="high",
    start_date=TRADE_START,
    end_date=TRADE_END,
)
dataset_spec = OptionMlDatasetSpec(
    rank_spec=OptionLabelSpec(label_method="rank", include_equity=False),
    mv_spec=OptionLabelSpec.diversified_mean_variance(include_equity=False),
    hybrid_spec=OptionLabelSpec.diversified_hybrid(include_equity=False),
    thetadata=ThetaDataDownloadSpec(max_dte=MAX_DTE, strike_range=STRIKE_RANGE),
    download_missing=DOWNLOAD_MISSING_SNAPSHOTS,
)

display(prices.tail())


In [ ]:
trade_result = build_trade_results([SYMBOL], spec=label_spec, price_frames={SYMBOL: prices})
trades = pd.DataFrame(trade_result.completed_trades)
if trades.empty:
    raise RuntimeError("No oracle AAPL trades were generated from Quant Warehouse prices")

display(trades.head(12))
display(
    trades.groupby(["freq", "k", "side"], dropna=False)
    .size()
    .rename("trades")
    .reset_index()
    .sort_values(["freq", "k", "side"])
)
print(f"oracle_trade_windows={len(trades)}")
print(f"unique_oracle_trades={trades['trade_id'].nunique() if 'trade_id' in trades.columns else len(trades)}")


In [ ]:
option_result = build_option_ml_dataset(trades, dataset_spec=dataset_spec)
panel = pd.DataFrame(option_result.rows)

fallback_path = warehouse_root / "research" / "options_eda_output" / "aapl_option_ml_dataset.parquet"
source_name = "rebuild"
if panel.empty and fallback_path.exists():
    panel = pd.read_parquet(fallback_path)
    source_name = str(fallback_path)

if panel.empty:
    raise RuntimeError("No option rows were available from rebuild or fallback sample data")

panel = panel.loc[panel["label_method"].eq("rank")].copy()
if "is_equity" in panel.columns:
    panel = panel.loc[~panel["is_equity"].astype(bool)].copy()

panel = panel.sort_values(["trade_id", "rank_y"], ascending=[True, False])
panel.to_parquet(ARTIFACT_DIR / "aapl_option_training_panel.parquet", index=False)

display(panel.head(12))
print(f"panel_source={source_name}")
print(f"panel_rows={len(panel)}")
print(f"panel_trade_ids={panel['trade_id'].nunique() if 'trade_id' in panel.columns else 0}")
print(option_result.statistics if option_result.statistics else {})


In [ ]:
def add_entry_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    numeric_candidates = [
        "bid_entry",
        "ask_entry",
        "entry_quote",
        "trade_duration_days",
        "k",
        "strike_entry",
        "volume_entry",
        "count_entry",
        "bid_size_entry",
        "ask_size_entry",
        "open_price_entry",
        "high_price_entry",
        "low_price_entry",
        "last_trade_price_entry",
    ]
    for col in numeric_candidates:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if {"bid_entry", "ask_entry"}.issubset(out.columns):
        out["entry_mid"] = (out["bid_entry"] + out["ask_entry"]) / 2.0
        out["entry_spread"] = out["ask_entry"] - out["bid_entry"]
        out["entry_spread_pct"] = out["entry_spread"] / out["entry_mid"].replace(0, np.nan)
    else:
        out["entry_mid"] = np.nan
        out["entry_spread"] = np.nan
        out["entry_spread_pct"] = np.nan

    if {"expiration_entry", "snapshot_date_entry"}.issubset(out.columns):
        expiration = pd.to_datetime(out["expiration_entry"], errors="coerce")
        snapshot = pd.to_datetime(out["snapshot_date_entry"], errors="coerce")
        out["days_to_expiry"] = (expiration - snapshot).dt.days
        out["snapshot_dow"] = snapshot.dt.dayofweek
        out["snapshot_month"] = snapshot.dt.month
    else:
        out["days_to_expiry"] = np.nan
        out["snapshot_dow"] = np.nan
        out["snapshot_month"] = np.nan

    return out


training = add_entry_features(panel)
training = training.loc[training["target_value"].notna()].copy()
training["target"] = pd.to_numeric(training["target_value"], errors="coerce")
training = training.loc[training["target"].notna()].copy()

leakage_cols = {
    "target",
    "target_value",
    "label",
    "rank_y",
    "rank_order",
    "mv_mu",
    "mv_weight",
    "option_return_pct",
    "underlying_return_pct",
    "entry_quote",
    "exit_quote",
    "trade_id",
    "contract_symbol",
    "target_col",
    "label_method",
    "task_name",
    "entry_snapshot_date",
    "exit_snapshot_date",
    "trade_entry_date",
    "trade_exit_date",
}

feature_df = training.drop(columns=[col for col in leakage_cols if col in training.columns])
feature_df = feature_df.drop(columns=[col for col in feature_df.columns if col.endswith("_exit")], errors="ignore")
feature_df = feature_df.dropna(axis=1, how="all")
feature_df = feature_df.loc[:, [col for col in feature_df.columns if feature_df[col].nunique(dropna=True) > 1]]
for col in ("created_at_entry", "last_trade_time_entry", "snapshot_date_entry", "expiration_entry"):
    if col in feature_df.columns:
        feature_df = feature_df.drop(columns=[col])

categorical_cols = [
    col
    for col in ("option_type_entry", "side", "freq", "underlying_symbol", "option_type", "underlying_symbol_entry")
    if col in feature_df.columns
]
numeric_cols = [col for col in feature_df.columns if col not in categorical_cols]
for col in numeric_cols:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce")

X = feature_df
y = training["target"].astype(float)
groups = training["trade_id"] if "trade_id" in training.columns else pd.Series(np.arange(len(training)), index=training.index)

print(f"feature_rows={len(X)}")
print(f"feature_cols={len(X.columns)}")
print(f"categorical_cols={categorical_cols}")
print(f"numeric_cols_sample={numeric_cols[:12]}")


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_cols,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

regressor = RandomForestRegressor(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", regressor),
])

training = training.copy()
training["oof_pred"] = np.nan

n_groups = int(groups.nunique())
if n_groups >= 2:
    n_splits = min(5, n_groups)
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        training.iloc[test_idx, training.columns.get_loc("oof_pred")] = pipeline.predict(X.iloc[test_idx])
        print(f"fold={fold} train_rows={len(train_idx)} test_rows={len(test_idx)}")
    oof_mae = mean_absolute_error(y, training["oof_pred"])
    oof_r2 = r2_score(y, training["oof_pred"])
else:
    pipeline.fit(X, y)
    training["oof_pred"] = pipeline.predict(X)
    oof_mae = mean_absolute_error(y, training["oof_pred"])
    oof_r2 = float("nan") if len(training) < 2 else r2_score(y, training["oof_pred"])

def _top_hit_rate(frame: pd.DataFrame, k: int = 1) -> float:
    if frame.empty or "trade_id" not in frame.columns:
        return float("nan")
    hits = []
    for _, group in frame.groupby("trade_id"):
        if group.empty or "oof_pred" not in group.columns:
            continue
        actual_best = group.loc[group["option_return_pct"].astype(float).idxmax(), "contract_symbol"] if "option_return_pct" in group.columns else None
        if actual_best is None:
            continue
        predicted_top = group.sort_values("oof_pred", ascending=False).head(k)["contract_symbol"].tolist()
        hits.append(actual_best in predicted_top)
    return float(np.mean(hits)) if hits else float("nan")

def _trade_eval_rows(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if frame.empty or "trade_id" not in frame.columns:
        return pd.DataFrame(rows)
    for trade_id, group in frame.groupby("trade_id"):
        if group.empty:
            continue
        best_actual_idx = group["option_return_pct"].astype(float).idxmax() if "option_return_pct" in group.columns else group["target"].astype(float).idxmax()
        best_pred_idx = group["oof_pred"].astype(float).idxmax()
        best_actual = group.loc[best_actual_idx]
        best_pred = group.loc[best_pred_idx]
        rows.append(
            {
                "trade_id": trade_id,
                "actual_best_contract": best_actual.get("contract_symbol"),
                "pred_best_contract": best_pred.get("contract_symbol"),
                "hit": bool(best_actual.get("contract_symbol") == best_pred.get("contract_symbol")),
                "actual_best_return_pct": pd.to_numeric(best_actual.get("option_return_pct"), errors="coerce"),
                "pred_best_return_pct": pd.to_numeric(best_pred.get("option_return_pct"), errors="coerce"),
                "actual_best_rank": pd.to_numeric(best_actual.get("target"), errors="coerce"),
                "pred_best_rank": pd.to_numeric(best_pred.get("target"), errors="coerce"),
                "pred_score": pd.to_numeric(best_pred.get("oof_pred"), errors="coerce"),
            }
        )
    return pd.DataFrame(rows)

trade_eval = _trade_eval_rows(training)

metrics = {
    "rows": int(len(training)),
    "trade_ids": int(training["trade_id"].nunique()) if "trade_id" in training.columns else 0,
    "features": int(X.shape[1]),
    "oof_mae": float(oof_mae),
    "oof_r2": None if pd.isna(oof_r2) else float(oof_r2),
    "top1_hit_rate": None if trade_eval.empty else float(trade_eval["hit"].mean()),
    "top3_hit_rate": _top_hit_rate(training, k=3),
    "avg_selected_return_pct": None if trade_eval.empty else float(pd.to_numeric(trade_eval["pred_best_return_pct"], errors="coerce").mean()),
    "avg_oracle_best_return_pct": None if trade_eval.empty else float(pd.to_numeric(trade_eval["actual_best_return_pct"], errors="coerce").mean()),
    "avg_return_gap_pct": None
    if trade_eval.empty
    else float((pd.to_numeric(trade_eval["pred_best_return_pct"], errors="coerce") - pd.to_numeric(trade_eval["actual_best_return_pct"], errors="coerce")).mean()),
}

print(json.dumps(metrics, indent=2))
display(trade_eval.head(12))
training.loc[:, [c for c in ["trade_id", "contract_symbol", "target", "oof_pred", "option_return_pct"] if c in training.columns]].head(12)


In [ ]:
pipeline.fit(X, y)
training["pred_score"] = pipeline.predict(X)

model_artifact = ARTIFACT_DIR / "aapl_option_picker.joblib"
summary_artifact = ARTIFACT_DIR / "aapl_option_picker_summary.json"
predictions_artifact = ARTIFACT_DIR / "aapl_option_picker_scored_rows.parquet"

joblib.dump(
    {
        "symbol": SYMBOL,
        "label_spec": asdict(label_spec),
        "dataset_spec": {
            "rank_spec": asdict(dataset_spec.rank_spec),
            "mv_spec": asdict(dataset_spec.mv_spec),
            "hybrid_spec": asdict(dataset_spec.hybrid_spec),
            "thetadata": asdict(dataset_spec.thetadata),
            "download_missing": dataset_spec.download_missing,
        },
        "feature_columns": list(X.columns),
        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
        "pipeline": pipeline,
        "metrics": metrics,
    },
    model_artifact,
)
summary_artifact.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
training.to_parquet(predictions_artifact, index=False)

print(f"saved_model={model_artifact}")
print(f"saved_summary={summary_artifact}")
print(f"saved_scored_rows={predictions_artifact}")


In [ ]:
latest_trade_id = training["trade_id"].dropna().iloc[-1]
latest_trade = training.loc[training["trade_id"].eq(latest_trade_id)].copy()
latest_trade["pred_score"] = pipeline.predict(latest_trade[X.columns])

score_cols = [
    col
    for col in ["contract_symbol", "option_type_entry", "strike_entry", "bid_entry", "ask_entry", "option_return_pct", "target", "pred_score"]
    if col in latest_trade.columns
]

display(latest_trade.sort_values("pred_score", ascending=False).loc[:, score_cols].head(15))
print(f"latest_trade_id={latest_trade_id}")
print(f"latest_trade_contracts={len(latest_trade)}")
